In [ ]:
print("Prepering dependecies.")
!pip -q install --upgrade unsloth trl peft accelerate bitsandbytes
print("Dependencies installed.")

Prepering dependecies.
Dependencies installed.


In [ ]:
import torch

print("=== Runtime info ===")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU 0 memory (GB): {props.total_memory / 1e9:.2f}")
else:
    print("GPU: None")

=== Runtime info ===
PyTorch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU 0: Tesla T4
GPU 0 memory (GB): 15.64


In [ ]:
from pathlib import Path
import os

# Needed for continual pretraining (disables CCE which is unsupported for CPT).
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

PRE_TRAIN_DIR = Path("/content/pre-train.json")
FINE_TUNE_DIR = Path("/content/train.json")

print(f"Pre-train path: {PRE_TRAIN_DIR} (exists={PRE_TRAIN_DIR.exists()})")
print(f"Fine-tune path: {FINE_TUNE_DIR} (exists={FINE_TUNE_DIR.exists()})")

Pre-train path: /content/pre-train.json (exists=True)
Fine-tune path: /content/train.json (exists=True)


In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Ministral-3-3B-Instruct-2512"
max_seq_length = 1024
dtype = None

print("Loading model...")
print(f"Model: {model_name}")
print(f"Max seq length: {max_seq_length}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
 )
print("Model loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model...
Model: unsloth/Ministral-3-3B-Instruct-2512
Max seq length: 1024
==((====))==  Unsloth 2026.5.2: Fast Ministral3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/976 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

Model loaded.


In [ ]:
from datasets import load_dataset

print("Loading datasets...")
pre_train_dataset = load_dataset("json", data_files=str(PRE_TRAIN_DIR), split="train")
fine_tune_dataset = load_dataset("json", data_files=str(FINE_TUNE_DIR), split="train")

print(f"Pre-training dataset size: {len(pre_train_dataset)}")
print(f"Pre-training columns: {pre_train_dataset.column_names}")
print(f"Fine-tuning dataset size: {len(fine_tune_dataset)}")
print(f"Fine-tuning columns: {fine_tune_dataset.column_names}")

Loading datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Pre-training dataset size: 111899
Pre-training columns: ['sentence']
Fine-tuning dataset size: 93521
Fine-tuning columns: ['instruction', 'input', 'output', 'text']


In [ ]:
print("Configuring LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
)
print("LoRA configured.")

Configuring LoRA...
LoRA configured.


In [ ]:
#print("Saving initial LoRA checkpoint...")
#model.save_pretrained("initial_model_checkpoint")
#tokenizer.save_pretrained("initial_model_checkpoint")
#print("Initial checkpoint saved: initial_model_checkpoint")

In [ ]:
from unsloth import FastLanguageModel

# This cell was causing state issues by merging weights prematurely.
# We will disable the 'sanity check' export here to prevent the LoRA mismatch error later.
print("Skipping GGUF sanity check to preserve LoRA adapter state for training...")
# FastLanguageModel.for_training(model)
# model.save_pretrained_gguf("test_gguf_before_training", tokenizer, quantization_method="q4_k_m")

Skipping GGUF sanity check to preserve LoRA adapter state for training...


In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments

EOS_TOKEN = tokenizer.eos_token

def formatting_pretrain_func(examples):
    sentences = examples["sentence"]
    if isinstance(sentences, list):
        return {"text": [f"{s}{EOS_TOKEN}" for s in sentences]}
    return {"text": [f"{sentences}{EOS_TOKEN}"]}

print("Formatting pre-training dataset...")
pre_train_dataset = pre_train_dataset.map(formatting_pretrain_func, batched=True)
print("Pre-training dataset formatted.")

print("Setting up pre-training trainer...")
trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=pre_train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        embedding_learning_rate=1e-5,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_pretrain",
        report_to="none",
        save_steps=200,
    ),
)
print("Pre-training trainer ready.")

Setting up pre-training trainer...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/111899 [00:00<?, ? examples/s]

Pre-training trainer ready.


In [ ]:
print("Starting pre-training...")
trainer_stats = trainer.train()
print("Pre-training complete.")

Starting pre-training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 111,899 | Num Epochs = 1 | Total steps = 13,988
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 67,502,080 of 3,916,592,128 (1.72% trained)


Step,Training Loss
1,2.906945
2,2.794114
3,2.924173
4,2.929718
5,3.206030
6,2.892417
7,2.614056
8,2.667206
9,2.965508
10,2.810511


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs_pretrain/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_pretrain/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_pretrain/checkpoint-600/tokenizer_config.json.


In [ ]:
# Fine-tuning phase (instruction tuning)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = (
            "Below is an instruction that describes a task, paired with an input that provides "
            "further context. Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input}\n\n"
            f"### Response:\n{output}{tokenizer.eos_token}"
        )
        texts.append(text)
    return {"text": texts}

print("Formatting fine-tune dataset...")
fine_tune_dataset = fine_tune_dataset.map(formatting_prompts_func, batched=True)
print("Fine-tune dataset formatted.")

print("Setting up fine-tuning trainer...")
trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=fine_tune_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        embedding_learning_rate=1e-5,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_finetune",
        report_to="none",
        save_steps=200,
    ),
)

In [ ]:
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

In [ ]:
import torch
FastLanguageModel.for_inference(model)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device for inference: {device}")

def generate_response(prompt_text):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant."}]},
        {"role": "user", "content": [{"type": "text", "text": prompt_text}]},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

print("Test 1 (Capital):")
print(generate_response("Care-i capitala Franței?"))
print("-" * 30)

print("Test 2 (Recipe):")
print(generate_response("Dă-mi o rețetă de gogoși cu prune."))

In [ ]:
from pathlib import Path
from unsloth import FastLanguageModel

print("Saving LoRA and GGUF outputs...")
FastLanguageModel.for_training(model)

output_root = Path("outputs")
output_root.mkdir(exist_ok=True)

lora_dir = output_root / "lora_model"
gguf_dir = output_root / "gguf_model"

model.save_pretrained(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))

try:
    gguf_model, gguf_tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(lora_dir),
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=True,
    )
    gguf_model.save_pretrained_gguf(
        str(gguf_dir),
        gguf_tokenizer,
        quantization_method="q4_k_m",
    )
    print(f"GGUF saved: {gguf_dir}")
except Exception as exc:
    print(f"GGUF export failed: {exc}")

print(f"LoRA saved: {lora_dir}")

In [ ]:
import os

try:
    from google.colab import files
except Exception:
    files = None

gguf_files = [f for f in os.listdir("gguf_model") if f.endswith(".gguf")]
if not gguf_files:
    print("No .gguf files found in gguf_model.")
elif files is None:
    print("gguf_model created, but google.colab.files is unavailable in this environment.")
    print("Manually download the .gguf file from gguf_model.")
else:
    gguf_file = os.path.join("gguf_model", gguf_files[0])
    print(f"Downloading: {gguf_file}")
    files.download(gguf_file)